# 2.8 Data and prediction drift

Data drift, prediction drift, choosing drift methods and thresholds, and a custom drift method with Evidently 0.7.

In [ ]:
import warnings

import pandas as pd
from sklearn import datasets, ensemble

from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset

# ValueDrift = drift for one column; DriftedColumnsCount = how many columns drifted
from evidently.metrics import DriftedColumnsCount, ValueDrift
from evidently.tests import lte

warnings.filterwarnings("ignore")

## Data and model

In [ ]:
# Adult (census income): numerical + categorical columns, target = income >50K
adult = datasets.fetch_openml(name="adult", version=2, as_frame=True).frame

numerical = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
categorical = ["workclass", "marital-status", "occupation", "relationship", "race", "sex"]
features = numerical + categorical

# Reference = younger people. Current = older people (drift) or another young sample (no drift)
young = adult[adult.age < 40]
reference = young.sample(5000, random_state=0)
current_drift = adult[adult.age >= 40].sample(5000, random_state=0)
current_same = young.drop(reference.index).sample(5000, random_state=1)
train = young.drop(reference.index).drop(current_same.index)

In [ ]:
# Train on separate data. Predictions on training rows are overconfident
# and would look like prediction drift against any new batch.
X_ref = pd.get_dummies(train[features])
model = ensemble.RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1)
model.fit(X_ref, train["class"] == ">50K")

def predict(df):
    X = pd.get_dummies(df[features]).reindex(columns=X_ref.columns, fill_value=0)
    return model.predict_proba(X)[:, 1]   # probability of >50K

for df in (reference, current_drift, current_same):
    df["prediction"] = predict(df)

In [ ]:
definition = DataDefinition(numerical_columns=numerical + ["prediction"], categorical_columns=categorical)

def to_ds(df):
    return Dataset.from_pandas(df[features + ["prediction"]], data_definition=definition)

ref_ds, drift_ds, same_ds = to_ds(reference), to_ds(current_drift), to_ds(current_same)

## Data drift report

`DataDriftPreset` tests every column with the default method for its type and size,
then flags **dataset drift** if ≥ 50% of columns drifted (`drift_share=0.5`).

In [ ]:
drift_result = Report([DataDriftPreset()]).run(current_data=drift_ds, reference_data=ref_ds)
drift_result

In [ ]:
# Same check with no real change – should be (almost) no drift
same_result = Report([DataDriftPreset()]).run(current_data=same_ds, reference_data=ref_ds)
same_result.dict()["metrics"][0]["value"]

In [ ]:
# Per-column scores as a table. Default here (>1000 rows) = distances: drift if score >= threshold
def drift_table(result):
    rows = []
    for m in result.dict()["metrics"]:
        if m["config"]["type"].endswith("ValueDrift"):
            rows.append({"column": m["config"]["column"], "score": round(m["value"], 3), "metric": m["metric_name"]})
    return pd.DataFrame(rows).sort_values("score", ascending=False)

drift_table(drift_result)

## Prediction drift

The best proxy when labels are delayed: did the model's outputs change?

In [ ]:
pred_drift = Report([ValueDrift(column="prediction")])
for name, ds in [("drift", drift_ds), ("same", same_ds)]:
    m = pred_drift.run(current_data=ds, reference_data=ref_ds).dict()["metrics"][0]
    print(name.ljust(6), m["metric_name"], "score =", round(m["value"], 3))

## Choosing methods and thresholds

Method names: `ks`, `chisquare`, `z`, `wasserstein`, `jensenshannon`, `psi`, `kl_div`, `hellinger`, `anderson`, ...

- `p-value` tests: drift if score **<** threshold
- distances: drift if score **≥** threshold

In [ ]:
# Small samples -> statistical tests make sense
small_ref = Dataset.from_pandas(reference.sample(500, random_state=0)[features + ["prediction"]], data_definition=definition)
small_cur = Dataset.from_pandas(current_drift.sample(500, random_state=0)[features + ["prediction"]], data_definition=definition)

methods = Report([
    ValueDrift(column="age", method="psi", threshold=0.2),        # PSI: > 0.2 is a common "big shift"
    ValueDrift(column="sex", method="jensenshannon"),
    ValueDrift(column="hours-per-week", method="ks"),
    # Methods per type + alert if more than 30% of columns drift
    DriftedColumnsCount(num_method="ks", cat_method="chisquare", share_tests=[lte(0.3)]),
])
methods_result = methods.run(current_data=small_cur, reference_data=small_ref)
methods_result

In [ ]:
# Per-column settings inside the preset: different method/threshold for one important feature
tuned = Report([DataDriftPreset(
    num_method="wasserstein", num_threshold=0.2,
    per_column_method={"age": "psi"}, per_column_threshold={"age": 0.25},
)])
tuned.run(current_data=drift_ds, reference_data=ref_ds).dict()["metrics"][0]["value"]

## Custom drift method

Any function `(reference, current, feature_type, threshold) -> (score, drifted)`.
Here: how many reference standard deviations the mean moved.

In [ ]:
from evidently.legacy.calculations.stattests import StatTest
from evidently.legacy.calculations.stattests.registry import register_stattest
from evidently.legacy.core import ColumnType

def _mean_shift(reference_data, current_data, feature_type, threshold):
    score = abs(current_data.mean() - reference_data.mean()) / reference_data.std()
    return score, score >= threshold

mean_shift = StatTest(
    name="mean_shift",
    display_name="Mean shift (in std)",
    allowed_feature_types=[ColumnType.Numerical],
    default_threshold=0.5,
)
register_stattest(mean_shift, default_impl=_mean_shift)

# After registering, use it by name like any built-in method
custom = Report([DataDriftPreset(columns=numerical, num_method="mean_shift")])
custom_result = custom.run(current_data=drift_ds, reference_data=ref_ds)
drift_table(custom_result)

## Output

In [ ]:
drift_result.save_html("data_drift_report.html")